# SpeechBrain SSL HuBERT-style Training in Google Colab

This notebook trains a HuBERT-style SSL model using SpeechBrain. It's adapted from `runyoro_speech_ai/speechbrain_ssl_training/train_sb_ssl.py`.

## Setup

1.  **Runtime**: Make sure your Google Colab runtime is set to GPU (Runtime > Change runtime type > GPU).
2.  **Clone Repository**: If you haven't already, clone your repository and change into its directory.
    ```bash
    # !git clone <your_repo_url>
    # %cd <your_repo_name>
    ```
3.  **Install Dependencies**: The cell below will install dependencies from `requirements.txt`.

In [4]:
# Install dependencies
!pip install -r requirements.txt
# SpeechBrain often has specific version requirements or extras.
# If issues arise, you might need to install a specific version or extras like:
# !pip install speechbrain==<version>
# !pip install speechbrain[optional_libs]
!pip install speechbrain==0.5.14 # Replace with your target version
!pip install torch==1.12.1+cu113 torchaudio==0.12.1+cu113 --extra-index-url https://download.pytorch.org/whl/cu113
!pip install transformers==4.20.0 # Example, adjust if used
!pip install PyYAML

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.3/174.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 864.1/864.1 kB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/

/content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai


## Configuration

This section requires careful setup of paths on your Google Drive.

-   **`BASE_DRIVE_PATH`**: Path to your project's root folder on Google Drive.
-   **`HPARAMS_FILE_REL_PATH`**: Relative path *within your project repo* to the `hparams_ssl.yaml` file. This script will copy it to your Drive output folder for this experiment.
-   **`OUTPUT_FOLDER_DRIVE`**: Path on Google Drive where checkpoints, logs, and the copied hparams for this experiment will be saved.
-   **`DATA_FOLDER_DRIVE`**: Path on Google Drive where your main data (e.g., audio files, manifests) is stored. This will be used to override `data_folder` in the YAML.
-   **`TRAIN_MANIFEST_REL_PATH`**: Relative path *within `DATA_FOLDER_DRIVE`* to your training manifest JSON file.
-   **`KMEANS_TARGET_DIR_REL_PATH`**: Relative path *within `OUTPUT_FOLDER_DRIVE`* where k-means targets are expected or will be generated. The `hparams_ssl.yaml` `target_label_dir` will be dynamically overridden to point here.

**Example Structure on Google Drive:**
```
MyDrive/
└── your_project_repo_on_drive/  (<- BASE_DRIVE_PATH)
    ├── runyoro_speech_ai/
    │   └── speechbrain_ssl_training/
    │       └── hparams_ssl.yaml (<- HPARAMS_FILE_REL_PATH leads here from project root)
    ├── colab_experiments/
    │   └── ssl_training_run_01/ (<- OUTPUT_FOLDER_DRIVE could be this)
    │       ├── logs/
    │       ├── saved_checkpoints/
    │       └── hparams_ssl_exp.yaml (copied and modified hparams)
    │       └── kmeans_frame_labels/ (<- KMEANS_TARGET_DIR_REL_PATH would be 'kmeans_frame_labels/')
    └── data_for_colab/ (<- DATA_FOLDER_DRIVE)
        ├── train_manifest.json (<- TRAIN_MANIFEST_REL_PATH)
        └── audio_files/
            └── ...
```
**Important**: The paths in your `hparams_ssl.yaml` (like `train_sb_manifest_file`, `data_folder`, `output_folder`, `target_label_dir`) will be overridden by the script based on the Colab Drive paths you define below.

In [5]:
# --- Imports --------------------------------------------------------------
import os, yaml, torch
from speechbrain.utils.hpopt import load_hyperpyyaml   # always available in SB ≥0.5

# --- User configuration ---------------------------------------------------
COLAB_PROJECT_ROOT   = "/content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/"
BASE_DRIVE_PATH      = "/content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/"
HPARAMS_FILE_REL_PATH = "runyoro_speech_ai/speechbrain_ssl_training/hparams_ssl.yaml"
EXPERIMENT_NAME      = "ssl_experiment_colab_01"

DATA_FOLDER_DRIVE          = os.path.join(BASE_DRIVE_PATH, "youtube_ssl_data")
TRAIN_MANIFEST_REL_PATH    = "3_manifests/youtube_audio_manifest.json"
KMEANS_TARGET_DIR_REL_PATH = "5_kmeans_targets"
# --------------------------------------------------------------------------

# --- Derived paths --------------------------------------------------------
HPARAMS_FILE_SOURCE_PATH = os.path.join(COLAB_PROJECT_ROOT, HPARAMS_FILE_REL_PATH)
OUTPUT_FOLDER_DRIVE      = os.path.join(BASE_DRIVE_PATH, "colab_experiments", EXPERIMENT_NAME)
HPARAMS_FILE_EXP_PATH    = os.path.join(OUTPUT_FOLDER_DRIVE, "hparams_ssl_colab_exp.yaml")

TRAIN_MANIFEST_FULL_PATH_DRIVE = os.path.join(DATA_FOLDER_DRIVE, TRAIN_MANIFEST_REL_PATH)
KMEANS_TARGET_DIR_FULL_PATH_DRIVE = os.path.join(DATA_FOLDER_DRIVE, KMEANS_TARGET_DIR_REL_PATH)

for p in [OUTPUT_FOLDER_DRIVE, DATA_FOLDER_DRIVE, KMEANS_TARGET_DIR_FULL_PATH_DRIVE]:
    os.makedirs(p, exist_ok=True)

print("✅  All Drive directories ensured.\n")

# --- Load the original YAML (with !ref / !new tags) -----------------------
if not os.path.exists(HPARAMS_FILE_SOURCE_PATH):
    raise FileNotFoundError(f"Hparams file not found → {HPARAMS_FILE_SOURCE_PATH}")

with open(HPARAMS_FILE_SOURCE_PATH, "r") as fp:
    hparams_dict = load_hyperpyyaml(
        fp,
        overrides={
            "output_folder"        : OUTPUT_FOLDER_DRIVE,
            "data_folder"          : DATA_FOLDER_DRIVE,
            "target_label_dir"     : KMEANS_TARGET_DIR_FULL_PATH_DRIVE,
            "train_sb_manifest_file": TRAIN_MANIFEST_FULL_PATH_DRIVE,
        },
        overrides_must_match=False,   # relax structural check
    )

# --- (Optional) print to verify -----------------------------------------
for k in ("output_folder", "data_folder",
          "target_label_dir", "train_sb_manifest_file"):
    print(f"{k}: {hparams_dict[k]}")

# --- Patch the paths in memory -------------------------------------------
hparams_dict["data_folder"]      = DATA_FOLDER_DRIVE
hparams_dict["output_folder"]    = OUTPUT_FOLDER_DRIVE
hparams_dict["target_label_dir"] = KMEANS_TARGET_DIR_FULL_PATH_DRIVE

# adapt this if your YAML stores the manifest deeper
if "train_sb_manifest_file" in hparams_dict:
    hparams_dict["train_sb_manifest_file"] = TRAIN_MANIFEST_FULL_PATH_DRIVE
else:
    raise KeyError(
        "'train_sb_manifest_file' not found—open the YAML and set the right key here."
    )

# --- Save the patched YAML -----------------------------------------------
#with open(HPARAMS_FILE_EXP_PATH, "w") as fp:
#    yaml.dump(hparams_dict, fp, default_flow_style=False)

#print("📝  Patched hparams saved to:", HPARAMS_FILE_EXP_PATH, "\n")

# --- Args for the training script ----------------------------------------
colab_script_args = {
    "hparams_file" : HPARAMS_FILE_SOURCE_PATH,
    "output_folder": OUTPUT_FOLDER_DRIVE,
    "data_folder"  : DATA_FOLDER_DRIVE,
    "device"       : "cuda" if torch.cuda.is_available() else "cpu",
}
print("Training-script arguments:")
print(colab_script_args)


✅  All Drive directories ensured.



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


output_folder: /content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/colab_experiments/ssl_experiment_colab_01
data_folder: /content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/youtube_ssl_data
target_label_dir: /content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/youtube_ssl_data/5_kmeans_targets
train_sb_manifest_file: /content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/youtube_ssl_data/3_manifests/youtube_audio_manifest.json
Training-script arguments:
{'hparams_file': '/content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/runyoro_speech_ai/speechbrain_ssl_training/hparams_ssl.yaml', 'output_folder': '/content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/colab_experiments/ssl_experiment_colab_01', 'data_folder': '/content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/youtube_ssl_data', 'device': 'cuda'}


## Training Script Logic

The following cell contains the main logic from `runyoro_speech_ai/speechbrain_ssl_training/train_sb_ssl.py`, adapted to run in this notebook environment using the configurations from the cell above.

In [6]:
import sys
import torch
import torch.nn.functional as F
import speechbrain as sb
from speechbrain.dataio.dataset import DynamicItemDataset
from speechbrain.dataio.dataio import read_audio
from speechbrain.lobes.models.huggingface_wav2vec import HuggingFaceWav2Vec2
import os
import logging
import argparse
import yaml # Ensure PyYAML is imported for hparams loading in main logic cell
import numpy as np

logger = logging.getLogger(__name__)
if not logger.hasHandlers():
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s [%(levelname)s] %(name)s - %(module)s.%(funcName)s - %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
else:
    logger.setLevel(logging.INFO)

class SSLBrain(sb.Brain):
    def __init__(self, modules=None, opt_class=None, hparams=None, run_opts=None, checkpointer=None):
        super().__init__(modules, opt_class, hparams, run_opts, checkpointer)
        if hasattr(self.modules, 'wav2vec2_model') and hasattr(self.modules, 'ssl_head'):
            try:
                if hasattr(self.modules.wav2vec2_model, 'model') and \
                   hasattr(self.modules.wav2vec2_model.model, 'config') and \
                   hasattr(self.modules.wav2vec2_model.model.config, 'hidden_size'):
                    encoder_out_dim = self.modules.wav2vec2_model.model.config.hidden_size
                    current_ssl_head = self.modules.ssl_head
                    if isinstance(current_ssl_head, sb.nnet.linear.Linear):
                        if hasattr(current_ssl_head, 'layers') and len(current_ssl_head.layers) > 0 and \
                           isinstance(current_ssl_head.layers[0], torch.nn.Linear):
                            actual_linear_layer = current_ssl_head.layers[0]
                            if actual_linear_layer.in_features != encoder_out_dim :
                                logger.info(f"Dynamically re-initializing ssl_head. Original: {actual_linear_layer.in_features}, New: {encoder_out_dim}")
                                self.modules.ssl_head = sb.nnet.linear.Linear(
                                    input_size=encoder_out_dim,
                                    n_neurons=self.hparams.num_ssl_clusters,
                                    bias=True
                                ).to(self.device)
                            else:
                                logger.info(f"SSL_Head input ({actual_linear_layer.in_features}) matches Wav2Vec2 output ({encoder_out_dim}).")
                        else: logger.warning("Could not verify/set ssl_head input size dynamically.")
                else: logger.warning("Could not dynamically determine wav2vec2_model output size for ssl_head.")
            except Exception as e: logger.warning(f"Error dynamically setting/checking ssl_head input size: {e}")

        if hasattr(self.modules, 'wav2vec2_model') and hasattr(self.modules.wav2vec2_model, 'model'):
             feat_dim = self.modules.wav2vec2_model.model.config.hidden_size
             self.mask_embedding = torch.nn.Parameter(torch.FloatTensor(feat_dim).uniform_())
             logger.info(f"Initialized learnable mask embedding: {self.mask_embedding.shape}")
        else:
            logger.error("wav2vec2_model not available for mask_embedding.")
            self.mask_embedding = torch.nn.Parameter(torch.FloatTensor(1).uniform_())

    def mask_acoustic_features(self, features, wav_lens):
        batch_size, seq_len, _ = features.shape
        boolean_mask = torch.zeros((batch_size, seq_len), dtype=torch.bool, device=features.device)
        masked_features = features.clone()
        for i in range(batch_size):
            current_num_frames = int(torch.round(wav_lens[i] * seq_len))
            if current_num_frames == 0: continue
            num_frames_to_mask = int(current_num_frames * self.hparams.mask_prob)
            min_frames_for_min_spans = self.hparams.mask_min_spans * self.hparams.mask_length
            if num_frames_to_mask < min_frames_for_min_spans and current_num_frames >= min_frames_for_min_spans :
                num_frames_to_mask = min_frames_for_min_spans
            num_frames_to_mask = min(num_frames_to_mask, current_num_frames)
            masked_indices_count = 0; attempts = 0
            while masked_indices_count < num_frames_to_mask and attempts < current_num_frames * 5:
                span_start = torch.randint(0, current_num_frames - self.hparams.mask_length + 1, (1,)).item()
                if masked_indices_count + self.hparams.mask_length > num_frames_to_mask * 1.5 and masked_indices_count > 0:
                    attempts += 1; continue
                for j in range(self.hparams.mask_length):
                    idx_to_mask = span_start + j
                    if idx_to_mask < current_num_frames:
                        if not boolean_mask[i, idx_to_mask]:
                            boolean_mask[i, idx_to_mask] = True
                            masked_features[i, idx_to_mask, :] = self.mask_embedding
                            masked_indices_count += 1
                    if masked_indices_count >= num_frames_to_mask: break
                if masked_indices_count >= num_frames_to_mask: break
                attempts += 1
            if num_frames_to_mask > 0 and masked_indices_count == 0: logger.debug(f"Item {i}: Could not mask. Req: {num_frames_to_mask}, Actual: {current_num_frames}")
            elif masked_indices_count < num_frames_to_mask: logger.debug(f"Item {i}: Masked {masked_indices_count}/{num_frames_to_mask}. Actual: {current_num_frames}")
        return masked_features, boolean_mask

    def compute_forward(self, batch, stage):
        batch = batch.to(self.device); wavs, wav_lens = batch.sig
        kmeans_targets_padded, kmeans_target_lens_abs = batch.kmeans_targets
        encoder_output_features = self.modules.wav2vec2_model(wavs, wav_lens)
        masked_input_for_head, time_mask_indices = self.mask_acoustic_features(encoder_output_features, wav_lens)
        predictions_logits = self.modules.ssl_head(masked_input_for_head)
        return predictions_logits, time_mask_indices, kmeans_targets_padded, kmeans_target_lens_abs

    def compute_objectives(self, forward_outputs, batch, stage):
        predictions_logits, time_mask_indices, kmeans_targets_padded, kmeans_target_lens_abs = forward_outputs
        kmeans_targets_padded = kmeans_targets_padded.to(predictions_logits.device)
        pred_seq_len = predictions_logits.size(1); target_seq_len = kmeans_targets_padded.size(1)
        if pred_seq_len != target_seq_len:
            min_len = min(pred_seq_len, target_seq_len)
            predictions_logits = predictions_logits[:, :min_len, :]; kmeans_targets_padded = kmeans_targets_padded[:, :min_len]; time_mask_indices = time_mask_indices[:, :min_len]
        masked_logits = predictions_logits[time_mask_indices]
        masked_targets = kmeans_targets_padded[time_mask_indices]
        if masked_logits.nelement() == 0 or masked_targets.nelement() == 0:
            logger.warning("No masked frames for loss. Returning zero loss."); return torch.tensor(0.0, device=self.device, requires_grad=True)
        loss = F.cross_entropy(masked_logits.reshape(-1, self.hparams.num_ssl_clusters), masked_targets.reshape(-1))
        if stage != sb.Stage.TRAIN:
            with torch.no_grad():
                predicted_ids = torch.argmax(masked_logits, dim=-1)
                correct_predictions = (predicted_ids == masked_targets).sum().item(); total_masked = masked_targets.numel()
                self.last_batch_accuracy = correct_predictions / total_masked if total_masked > 0 else 0.0
        return loss

    def on_stage_start(self, stage, epoch):
        if stage != sb.Stage.TRAIN: self.accuracies_this_epoch = []

    def on_stage_end(self, stage, stage_loss, epoch):
        stage_name = stage.name.capitalize()
        if stage == sb.Stage.TRAIN: self.train_loss = stage_loss
        logger.info(f"Epoch {epoch}: {stage_name} Loss = {stage_loss:.4f}")
        if stage != sb.Stage.TRAIN:
            if hasattr(self, 'accuracies_this_epoch') and self.accuracies_this_epoch:
                avg_accuracy = sum(self.accuracies_this_epoch) / len(self.accuracies_this_epoch)
                logger.info(f"Epoch {epoch}: {stage_name} Average Accuracy = {avg_accuracy:.3f}")
            elif hasattr(self, 'last_batch_accuracy'): logger.info(f"Epoch {epoch}: {stage_name} Last Batch Accuracy = {self.last_batch_accuracy:.3f}")

def dataio_prepare(hparams):
    logger.info("Preparing datasets for HuBERT-style SSL training...")
    target_label_dir = hparams["target_label_dir"]
    logger.info(f"K-means target labels expected from: {target_label_dir}")
    if not os.path.isdir(target_label_dir): logger.warning(f"K-means target label directory DOES NOT exist: {target_label_dir}. This WILL error if labels are needed now.")

    @sb.utils.data_pipeline.takes("wav")
    @sb.utils.data_pipeline.provides("sig")
    def audio_pipeline(wav_path):
        try: return read_audio(wav_path)
        except Exception as e: logger.error(f"Error reading audio {wav_path}: {e}", exc_info=True); raise

    @sb.utils.data_pipeline.takes("id")
    @sb.utils.data_pipeline.provides("kmeans_targets")
    def label_pipeline(utt_id):
        label_filename = f"{utt_id}_kmeans_labels.npy"
        label_path = os.path.join(target_label_dir, label_filename)
        try:
            targets = np.load(label_path); return torch.from_numpy(targets).long()
        except FileNotFoundError: logger.error(f"K-means target file not found: {label_path}."); raise
        except Exception as e: logger.error(f"Error loading K-means target {label_path}: {e}", exc_info=True); raise

    datasets_map = {}
    data_info = {"train": hparams["train_sb_manifest_file"]}

    for dataset_name, manifest_file_path in data_info.items():
        if not manifest_file_path or not os.path.exists(manifest_file_path):
            logger.warning(f"Manifest for '{dataset_name}' not found at {manifest_file_path} or path empty. Skipping.")
            continue
        logger.info(f"Loading '{dataset_name}' dataset from: {manifest_file_path}")
        datasets_map[dataset_name] = DynamicItemDataset.from_json(
            json_path=manifest_file_path,
            replacements={"data_root": hparams.get("data_folder")},
            dynamic_items=[audio_pipeline, label_pipeline],
            output_keys=["id", "sig", "kmeans_targets"],
        )
        logger.info(f"'{dataset_name}' dataset loaded. Samples: {len(datasets_map[dataset_name])}")

    if not datasets_map or "train" not in datasets_map:
        # HPARAMS_FILE_EXP_PATH is not defined in this cell, but hparams_file_for_sb is (from main execution logic)
        # For a more robust error message, we might need to pass hparams_file_for_sb into dataio_prepare if we want to log it here
        raise ValueError(f"Training data failed. Check manifest ({hparams.get('train_sb_manifest_file', 'N/A')}) and data_folder ({hparams.get('data_folder', 'N/A')}) in hparams.")
    return datasets_map

logger.info("SSLBrain class and dataio_prepare function defined.")

INFO:__main__:SSLBrain class and dataio_prepare function defined.


## Run Training

This cell executes the SpeechBrain training process using the configurations defined above.
Make sure:
-   The `colab_script_args` dictionary in the **Configuration** cell is correctly populated with paths on your Google Drive.
-   Your data (audio files, manifest file, k-means targets if pre-generated) are in the correct locations on Google Drive.
-   The `hparams_ssl_colab_exp.yaml` file has been successfully created in your `OUTPUT_FOLDER_DRIVE`.

In [ ]:
# --- Main Execution for Colab ---
import yaml # Ensure yaml is imported for the new loading mechanism
import torch
from speechbrain.utils.hpopt import load_hyperpyyaml
from speechbrain.utils.checkpoints import Checkpointer

hparams_file_for_sb = (
    "/content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/"
    "runyoro_speech_ai/speechbrain_ssl_training/hparams_ssl.yaml"
)

with open(hparams_file_for_sb) as fp:
    hparams = load_hyperpyyaml(
        fp,
        overrides={
            "output_folder": OUTPUT_FOLDER_DRIVE,
            "data_folder"  : DATA_FOLDER_DRIVE,
            "target_label_dir": KMEANS_TARGET_DIR_FULL_PATH_DRIVE,
            "train_sb_manifest_file": TRAIN_MANIFEST_FULL_PATH_DRIVE,
        },
        overrides_must_match=False,
    )

print("✓ YAML parsed — total keys:", len(hparams))
print("   output_folder:", hparams["output_folder"])
print("   data_folder  :", hparams["data_folder"])
# ------------------------------------------------------------------

if "colab_script_args" not in globals():
    raise NameError("colab_script_args not defined. Please run the Configuration cell (Cell 6) first.")

logger.info(f"Starting SpeechBrain SSL training with effective arguments: {colab_script_args}")

# Convert dict to a Namespace object to mimic argparse.Namespace if SpeechBrain internals expect it
effective_args = argparse.Namespace(**colab_script_args)

# The hparams file at HPARAMS_FILE_EXP_PATH already has Drive paths resolved for key items.
# Overrides here would be for things like epoch count, batch size, or device if not set in config cell.
hparams_file_for_sb = effective_args.hparams_file
cli_overrides_for_sb = {
    "output_folder": effective_args.output_folder, # Ensure SB uses the correct experiment output folder
    "data_folder": effective_args.data_folder # Ensure SB uses the correct data folder
}
if hasattr(effective_args, 'number_of_epochs') and effective_args.number_of_epochs is not None:
    cli_overrides_for_sb['number_of_epochs'] = effective_args.number_of_epochs
if hasattr(effective_args, 'batch_size') and effective_args.batch_size is not None:
    cli_overrides_for_sb['batch_size'] = effective_args.batch_size


# valid kwargs for this SB version
_valid_keys = {
    "checkpoints_dir",
    "recoverables",
    "loadables",
    "path",
    "load_if_possible",
    "recoverables_to_load",
    "custom_load_action",
    "device",
}
if isinstance(hparams["checkpointer"], dict):
    ckpt_cfg = {k: v for k, v in hparams["checkpointer"].items() if k in _valid_keys}
    # ensure the directory is set
    ckpt_cfg.setdefault("checkpoints_dir", hparams["output_folder"])
    hparams["checkpointer"] = Checkpointer(**ckpt_cfg)

# ------------------------------------------------------------------
# Add ssl_head if it isn't in the YAML
if "ssl_head" not in hparams["modules"]:
    enc = hparams["modules"]["wav2vec2_model"]
    enc_dim = getattr(enc, "output_dim", enc.model.config.hidden_size)
    n_clusters = 100   # <-- put your real number here
    hparams["modules"]["ssl_head"] = torch.nn.Linear(enc_dim, n_clusters)
# ------------------------------------------------------------------

# Note: Key paths like 'train_sb_manifest_file' and 'target_label_dir'
# are expected to be correctly set within the hparams_file_for_sb
# (HPARAMS_FILE_EXP_PATH) by the logic in Cell 6 (Configuration cell).
# The 'output_folder' and 'data_folder' are also confirmed by overrides here.

sb.create_experiment_directory(
    experiment_directory=hparams["output_folder"],
    hyperparams_to_save=hparams_file_for_sb, # Save the already modified yaml for this run
    overrides=cli_overrides_for_sb
)

datasets = dataio_prepare(hparams)

run_opts = {"device": effective_args.device}
logger.info(f"Running on device: {run_opts['device']}")

ssl_brain = SSLBrain(
    modules=hparams["modules"],
    opt_class=lambda params: getattr(torch.optim, hparams["optimizer"].capitalize())(params, lr=hparams["lr_adam"]),
    hparams=hparams,
    run_opts=run_opts,
    checkpointer=hparams["checkpointer"],
)

train_dataloader_opts = hparams.get("train_dataloader_opts", {})
if "batch_size" not in train_dataloader_opts: train_dataloader_opts["batch_size"] = hparams["batch_size"]
if "num_workers" not in train_dataloader_opts and "num_workers" in hparams: train_dataloader_opts["num_workers"] = hparams.get("num_workers", 0)
elif "num_workers" not in train_dataloader_opts and "dataloader_num_workers" in hparams: train_dataloader_opts["num_workers"] = hparams.get("dataloader_num_workers", 0)

logger.info(f"Starting training. Effective batch size: {hparams['batch_size']} * {hparams['grad_accumulation_factor']}")

if "epoch_counter" not in hparams: hparams["epoch_counter"] = sb.utils.epoch_loop.EpochCounter(limit=hparams["number_of_epochs"])

ssl_brain.fit(
    epoch_counter=hparams["epoch_counter"],
    train_set=datasets["train"],
    train_loader_kwargs=train_dataloader_opts,
    # valid_set=datasets.get("valid"), # Add if you have validation
    # valid_loader_kwargs=hparams.get("valid_dataloader_opts", {})
)

logger.info(f"HuBERT-style SSL training finished. Checkpoints/logs in: {hparams['output_folder']}")

INFO:__main__:Starting SpeechBrain SSL training with effective arguments: {'hparams_file': '/content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/runyoro_speech_ai/speechbrain_ssl_training/hparams_ssl.yaml', 'output_folder': '/content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/colab_experiments/ssl_experiment_colab_01', 'data_folder': '/content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/youtube_ssl_data', 'device': 'cuda'}


✓ YAML parsed — total keys: 27
   output_folder: /content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/colab_experiments/ssl_experiment_colab_01
   data_folder  : /content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/youtube_ssl_data
speechbrain.core - Beginning experiment!
speechbrain.core - Experiment folder: /content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/colab_experiments/ssl_experiment_colab_01
__main__ - Preparing datasets for HuBERT-style SSL training...
__main__ - K-means target labels expected from: /content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/youtube_ssl_data/5_kmeans_targets
__main__ - Loading 'train' dataset from: /content/drive/MyDrive/Runyoro_AI_Project/open-runyoro-ai/youtube_ssl_data/3_manifests/youtube_audio_manifest.json
__main__ - 'train' dataset loaded. Samples: 19939
__main__ - Running on device: cuda
speechbrain.core - Info: grad_accumulation_factor arg from hparam file is used
speechbrain.core - 315.5M trainable parameters in SSLBra

  8%|▊         | 380/4985 [17:51<3:18:05,  2.58s/it, train_loss=4.56]

## Notes

- **Data**: Ensure your audio data, manifest files (`train_sb_manifest.json`), and k-means target labels (`<utt_id>_kmeans_labels.npy` files in `KMEANS_TARGET_DIR_FULL_PATH_DRIVE`) are correctly placed on Google Drive and paths in the **Configuration** cell are accurate.
- **`hparams_ssl.yaml`**: The original `hparams_ssl.yaml` from your repository is copied to your experiment's output folder on Drive (`hparams_ssl_colab_exp.yaml`) and key paths are modified for the Colab environment. You can further adjust parameters like `batch_size` or `number_of_epochs` in the **Configuration** cell (Cell 6) before running, or by editing the `hparams_ssl_colab_exp.yaml` directly on Drive for subsequent runs.
- **Resuming Training**: SpeechBrain's checkpointer should automatically resume from the latest checkpoint in `OUTPUT_FOLDER_DRIVE` if training is interrupted and re-run.
- **Output**: All outputs (checkpoints, logs, modified hparams) are saved to `OUTPUT_FOLDER_DRIVE` on your Google Drive.